In [ ]:
import os
import pandas as pd
import tempfile
import shutil
import subprocess
from pathlib import Path
import xml.etree.ElementTree as ET

# Path to your dataset
input_csv_path = r"D:\Android_Mobile_App\AndroidProject_dataset\Repo_List.csv"
output_csv_path = r"D:\Android_Mobile_App\AndroidProject_dataset\Repo_List_checked.csv"

# Read the repo list
df = pd.read_csv(input_csv_path)

# Add new columns
df["has_manifest"] = "no"
df["has_activity"] = "no"

# Loop over each repo
for idx, row in df.iterrows():
    repo_url = row.get("clone_url") or row.get("html_url")
    if not repo_url or not isinstance(repo_url, str):
        continue

    # Create a temp folder for cloning
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, tmpdir],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except subprocess.CalledProcessError:
            continue  # skip repos that fail to clone

        # Search for AndroidManifest.xml
        manifest_path = None
        for root, dirs, files in os.walk(tmpdir):
            if "AndroidManifest.xml" in files:
                manifest_path = os.path.join(root, "AndroidManifest.xml")
                df.at[idx, "has_manifest"] = "yes"
                break

        # Check if manifest contains at least one <activity>
        if manifest_path:
            try:
                tree = ET.parse(manifest_path)
                root = tree.getroot()
                ns = {'android': 'http://schemas.android.com/apk/res/android'}
                activities = root.findall(".//activity", namespaces=ns)
                if not activities:
                    # fallback if namespace doesn't work
                    activities = root.findall(".//activity")
                if activities:
                    df.at[idx, "has_activity"] = "yes"
            except Exception:
                continue  # skip XML parse errors

# Save the updated CSV
df.to_csv(output_csv_path, index=False)
print(f"✅ Done. Output saved to: {output_csv_path}")
